# 19 - Generate LLM Context from Well Logs

**Part B - Integrated & Advanced Workflows**

This lesson turns LAS data into compact, auditable JSON context packages organized by analysis category, well, formation, and lithology group. It uses only public `petrophys` APIs and bundled sample data, so it runs without machine-specific paths.

The generated JSON is model-neutral: use it with any LLM client, retrieval system, or review workflow.

## Learning goals

- Build or save single-well context in one line after reading a well.
- Generate versioned category/formation/lithology packages.
- Read ranked CSV or NPD/SODIR Excel tables directly.
- Enforce byte budgets and inspect provenance, warnings and limitations.
- Build a model-neutral request without making a network call.

## What the exporter produces

For each lithology group, the exporter writes separate views for hydrocarbon, saturation, permeability, porosity, numerical statistics, and lithology analysis. Each JSON file records the source well, formation interval, input logs, computed logs, statistics, analysis results, data completeness, and bad-hole QC.

Raw depth-by-depth traces are intentionally excluded by default. This keeps context smaller while retaining the statistics and provenance needed for interpretation and review.

In [1]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
import json

import pandas as pd
import petrophys as pp

print("petrophys version:", pp.__version__)
well = pp.read(pp.sample("30_6-5"))
context = pp.build_context(well)
assert context["schema_version"] == "1.0"
assert context["interpretation"] is None
with TemporaryDirectory() as directory:
    saved = pp.export_context(well, Path(directory) / "context.json")
    print("Raw well context:", saved.name, saved.stat().st_size, "UTF-8 bytes")

petrophys version: 3.0.0


Raw well context: context.json 3356 UTF-8 bytes


## 1. Configure package paths once

The single-well API above needs no geology files. The package example below groups results by formation, so it explicitly supplies a formation table. All generated files go below a fresh temporary run directory; no private workbook is needed.

In [2]:
WELL_NAME = "30_6-5"
run_directory = TemporaryDirectory(prefix="petrophys-context-demo-")
RUN_ROOT = Path(run_directory.name)
OUTPUT_DIR = RUN_ROOT / "context"  # Must not exist before export.
LAS_FILE = Path(pp.sample(WELL_NAME))
FORMATION_SOURCE = Path(pp.sample("formations_30_6-5"))
INPUT_DIR = LAS_FILE.parent
MINIMUM_GROUP_THICKNESS = 2.0
MAX_BYTES = 1_000_000

# Illustrative assumptions, not a validated reservoir calibration.
# Record both Rw and its reference temperature; do not tune them to force pay.
WORKFLOW_CONFIG = pp.DynamicWorkflowConfig(rw=0.070, rw_meas_temp_f=230.0)

print("LAS:", LAS_FILE.name)
print("Formations:", FORMATION_SOURCE.name)
print("Profile:", WORKFLOW_CONFIG.interpretation_profile)
print("Overrides:", {"rw": WORKFLOW_CONFIG.rw, "rw_meas_temp_f": WORKFLOW_CONFIG.rw_meas_temp_f})

LAS: 30_6-5.las
Formations: formations_30_6-5.csv
Profile: generic_inference
Overrides: {'rw': 0.07, 'rw_meas_temp_f': 230.0}


For your own data, replace the LAS and formation paths. CSV/TSV/Excel tables are accepted, including NPD/SODIR `Well identifier` / `Surface` / `MD` headers. Supply sheet and depth metadata when needed. Codebooks are optional; ambiguous Kingdom label columns generate a warning and prefer `Attribute`.

## 2. Inspect the ranked formation table

The sample declares metre depths, explicit bottoms, rank and parent. Groups intentionally overlap their formations. Select `rank="formation"` for this package; do not infer hierarchy by deleting geometrically nested rows. No temporary tops workbook or guessed final bottom is needed.

In [3]:
formations = pp.read_formations(FORMATION_SOURCE)
display(formations[["formation_name", "top_depth", "bottom_depth", "rank", "parent"]])
assert set(formations["depth_unit"]) == {"m"}
assert {"group", "formation"}.issubset(set(formations["rank"]))
print("Selected formation intervals:", int((formations["rank"] == "formation").sum()))

,formation_name,top_depth,bottom_depth,rank,parent
0,NORDLAND GP.,181.0,726.0,group,
1,Utsira Fm.,726.0,879.0,formation,
2,HORDALAND GP.,879.0,1908.0,group,
3,ROGALAND GP.,1908.0,2203.0,group,
4,Balder Fm.,1908.0,2023.0,formation,ROGALAND GP.
5,Sele Fm.,2023.0,2123.0,formation,ROGALAND GP.
6,Lista Fm.,2123.0,2203.0,formation,ROGALAND GP.
7,SHETLAND GP.,2203.0,2408.0,group,
8,CROMER KNOLL GP.,2408.0,2452.0,group,
9,VIKING GP.,2452.0,2841.0,group,


Selected formation intervals: 12


## 3. Generate bounded, category-first context

Package exports run the workflow, group intervals, and validate all files before publishing the new directory. The versioned schema wraps each category document under `payload`. A failed or oversized export raises an exception without leaving a partial destination.

In [4]:
results = pp.export_context(
    INPUT_DIR, OUTPUT_DIR,
    layout="category_formation_lithology",
    tops_file=FORMATION_SOURCE,
    tops_options={"rank": "formation"},
    pattern=LAS_FILE.name,
    recursive=False,
    minimum_group_thickness=MINIMUM_GROUP_THICKNESS,
    well_names=[WELL_NAME],
    workflow_config=WORKFLOW_CONFIG,
    max_bytes=MAX_BYTES,
    max_total_bytes=50_000_000,
)
print("Generated records:", len(results))
assert results and all(item.success for item in results)

[repository]\src\petrophys\preprocessing\standardization\curve_standardizer.py:390: UserWarning: SWC: inferred percent core saturation despite unit 'frac'; divided by 100 to v/v. Verify source units.
  standardize_core_saturation(new_curve, inplace=True)
[repository]\src\petrophys\preprocessing\standardization\curve_standardizer.py:390: UserWarning: SOC: inferred percent core saturation despite unit 'frac'; divided by 100 to v/v. Verify source units.
  standardize_core_saturation(new_curve, inplace=True)


Generated records: 828


## 4. Summarize and review warnings

The API has checked failures. Still inspect warnings before interpreting the exported statistics; successful serialization is not scientific validation.

In [5]:
context_files = sorted({Path(item.json_file) for item in results})
category_counts = Counter(path.relative_to(OUTPUT_DIR).parts[0] for path in context_files)
summary = pd.DataFrame({
    "category": list(category_counts),
    "json_files": list(category_counts.values()),
}).sort_values("category", ignore_index=True)
display(summary)
warnings = sorted({warning for item in results for warning in item.warnings})
print("Export warnings:", len(warnings))
for warning in warnings[:5]:
    print(warning)
assert all(path.stat().st_size <= MAX_BYTES for path in context_files)
print("JSON context files:", len(context_files))

,category,json_files
0,hydrocarbon,138
1,lithology,138
2,numerical,138
3,permeability,138
4,porosity,138
5,saturation,138


Export warnings: 2
Auto lithology resolved via vendor_log (83% of 10732 samples): matrix_type=sandstone, lithology=clastic
Header parameter maps to rw=0.7 but the config sets it explicitly; config value kept.
JSON context files: 828


In [6]:
# Show a compact, relative file listing instead of machine-specific paths.
for path in context_files[:12]:
    print(path.relative_to(RUN_ROOT))
if len(context_files) > 12:
    print(f"... and {len(context_files) - 12} more JSON files")

context\hydrocarbon\30_6-5\AMUNDSEN FM\AMUNDSEN FM._SILT_GROUP_2_hydrocarbon.json
context\hydrocarbon\30_6-5\AMUNDSEN FM\AMUNDSEN FM._SILT_GROUP_4_hydrocarbon.json
context\hydrocarbon\30_6-5\AMUNDSEN FM\AMUNDSEN FM._SILT_GROUP_5_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_10_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_11_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_1_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_3_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_4_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_5_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_6_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._EXTRUSIVE_GROUP_7_hydrocarbon.json
context\hydrocarbon\30_6-5\BALDER FM\BALDER FM._SILT_GROUP_1_hydrocarbon.json
... and 81

## 5. Inspect one context document

Choose the category that matches the question being asked. Here we open a hydrocarbon context and inspect only high-signal metadata before using the complete payload.

In [7]:
hydrocarbon_files = [
    path for path in context_files
    if path.relative_to(OUTPUT_DIR).parts[0] == "hydrocarbon"
]
context_path = hydrocarbon_files[0] if hydrocarbon_files else context_files[0]
document = json.loads(context_path.read_text(encoding="utf-8"))
assert document["schema_version"] == "1.0"
payload = document["payload"]
analysis = payload["analysis_results"]
overview = pd.Series({
    "file": str(context_path.relative_to(RUN_ROOT)),
    "schema_version": document["schema_version"],
    "well": analysis["well_name"],
    "formation": analysis["formation_name"],
    "lithology_group": analysis["lithology group"],
    "lithology": analysis["lithology"],
    "input_log_count": len(analysis["input_logs_present"]),
    "derived_log_count": len(analysis["derived_logs_computed"]),
    "raw_traces_included": payload["include_raw_traces_in_context"],
})
display(overview.to_frame("value"))

,value
file,context\hydrocarbon\30_6-5\AMUNDSEN FM\AMUNDSE...
schema_version,1.0
well,30_6-5
formation,AMUNDSEN FM.
lithology_group,SILT_GROUP_2
lithology,SILT
input_log_count,14
derived_log_count,36
raw_traces_included,False


In [8]:
# Basic schema and provenance checks before an LLM sees the document.
required_top_level = {
    "formation_info",
    "analysis_results",
    "raw_log_traces_in_segment",
    "include_raw_traces_in_context",
}
required_analysis = {
    "well_name",
    "formation_name",
    "statistics",
    "analysis_summary",
    "data_completeness",
    "input_logs_present",
    "derived_logs_computed",
    "BADHOLE_QC_CHECK",
}

assert required_top_level.issubset(payload)
assert required_analysis.issubset(analysis)
assert analysis["well_name"]
assert payload["formation_info"]["thickness"] >= 0
print("Schema and provenance checks passed.")

Schema and provenance checks passed.


### Scientific review, beyond schema

Inspect a known interval and the assumptions used to calculate it. The following numerical checks only verify valid ranges and the explicit Rw basis. They do not establish hydrocarbons, validate the chosen calibration, or justify the shale/overburden pay flags. Review clay-volume methods, quality masks, saturation-model assumptions and `NET_PAY_REVIEW` before treating cutoff screening as pay.

In [9]:
etive_sand = [
    path for path in context_files
    if path.relative_to(OUTPUT_DIR).parts[0] == "saturation"
    and any("etive" in part.lower() for part in path.parts)
    and "SAND" in path.name
]
assert etive_sand, "No Etive sand context found; review input selection"
probe = json.loads(etive_sand[0].read_text(encoding="utf-8"))["payload"]
stats = probe["analysis_results"]["statistics"]
sw_mean = stats["SW_overall"]["arithmetic_mean"]
rw_mean = stats["RW_overall"]["arithmetic_mean"]
print(f"{etive_sand[0].name}")
print(f"SW mean: {sw_mean:.3f}; corrected Rw mean: {rw_mean:.3f} ohm.m")
assert 0 <= sw_mean <= 1
assert rw_mean > 0
print("Numerical checks passed; reservoir calibration still requires review.")

ETIVE FM._CROSSBEDDED SAND_GROUP_1_saturation.json
SW mean: 0.343; corrected Rw mean: 0.072 ohm.m
Numerical checks passed; reservoir calibration still requires review.


## 6. Build a model-neutral LLM request

Keep measured inputs, computed outputs, warnings, and missing-data statements intact. Ask the model to distinguish observation from inference and to avoid inventing values. The object below can be serialized for any LLM SDK or stored in a retrieval index.

In [10]:
llm_request = {
    "task": "Summarize the interval's measured evidence, calculations and limitations.",
    "instructions": [
        "Use only the supplied context.",
        "Separate measurements from calculated interpretations.",
        "Do not infer missing values or equate cutoff screening with confirmed pay.",
        "Report well, formation, lithology group and depth interval.",
        "Retain QC warnings and state which assumptions need calibration.",
    ],
    "petrophysical_context": document,
}
serialized_request = json.dumps(llm_request, ensure_ascii=False)
print("Request UTF-8 bytes:", len(serialized_request.encode("utf-8")))
print("Prepared locally; no LLM or network call was made.")
run_directory.cleanup()

Request UTF-8 bytes: 23256
Prepared locally; no LLM or network call was made.


## CLI equivalent

```bash
petrophys-context well.las context.json
petrophys-context logs outputs/context_run_01 --layout category_formation_lithology --formations tops.csv --tops-rank formation
```

Use the Python API's `workflow_config` for project-specific package assumptions. The historical scripts call the same API and require explicit input paths. `--schema-version legacy` returns the old payload structure; `--legacy-benchmark-compat` on those scripts also enables the archived benchmark conventions. Neither is needed for new interpretations.

## Recap

- `pp.build_context(well)` and `pp.export_context(well, path)` work without geology.
- Pass an interpretation result when calculated summaries are needed.
- Package exports accept real formation formats and explicit hierarchy selection.
- New exports are versioned, byte-bounded, and do not overwrite prior packages.
- Warnings and physical calibration matter beyond JSON/schema checks.
- Context generation is model-neutral and makes no LLM request.